# DBSCAN Basics: Understanding Density-Based Clustering

**Estimated Time:** 30-45 minutes  
**Difficulty:** Beginner to Intermediate

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the core concepts of density-based clustering
- Apply DBSCAN to various dataset types and shapes
- Interpret clustering results including core, border, and noise points
- Understand how DBSCAN parameters (eps, min_samples) affect results
- Recognize when DBSCAN is appropriate for your clustering task
- Visualize and analyze DBSCAN clustering outcomes

## Prerequisites

- Basic Python programming and NumPy
- Familiarity with Matplotlib for visualization
- Understanding of basic clustering concepts
- Completion of `00_quick_start.ipynb` (recommended)

## Paper References

This notebook introduces concepts from the foundational DBSCAN paper:
- **Section 1** (p. 226): Introduction and motivation
- **Section 4.1** (p. 227): Core concepts - epsilon-neighborhood, core points, density
- **Section 4.2** (p. 228): DBSCAN algorithm overview

**Full Reference:**  
Ester, M., Kriegel, H. P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. In *KDD* (Vol. 96, No. 34, pp. 226-231).

**Paper Location:** `../1996-DBSCAN-KDD.pdf` in repository root

---

## Table of Contents

1. [Conceptual Introduction](#conceptual-introduction)
2. [Mathematical Formalization](#mathematical-formalization)
3. [Setup and Imports](#setup)
4. [Basic DBSCAN Application](#basic-application)
5. [Understanding Point Types](#point-types)
6. [Parameter Effects](#parameter-effects)
7. [Different Dataset Shapes](#dataset-shapes)
8. [Exercises](#exercises)
9. [Summary](#summary)
10. [Next Steps](#next-steps)

---

## 1. Conceptual Introduction <a id='conceptual-introduction'></a>

### What is Density-Based Clustering?

Traditional clustering algorithms like K-Means assume clusters are spherical and require you to specify the number of clusters in advance. **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) takes a fundamentally different approach [Paper §1, p. 226].

### The Core Idea

DBSCAN identifies clusters as **dense regions** of points separated by regions of lower density. Think of it like finding islands of high population density on a map - the islands are your clusters, and the sparse ocean areas are noise.

**Key Insight**: Points that are close together and have many neighbors form clusters. Points in sparse regions are outliers (noise).

### Why DBSCAN?

DBSCAN offers several advantages:

1. **No need to specify number of clusters**: The algorithm discovers clusters automatically
2. **Arbitrary cluster shapes**: Can find non-spherical clusters (crescents, rings, etc.)
3. **Noise detection**: Automatically identifies outliers
4. **Deterministic**: Same input always produces same output
5. **Intuitive parameters**: Only two parameters with clear geometric meaning

### The Two Parameters [Paper §4.1, p. 227]

DBSCAN requires only two parameters:

1. **eps (ε)**: The maximum distance between two points to be considered neighbors
   - Think of it as the "radius of neighborhood"
   - Smaller ε → more strict about what counts as a neighbor
   - Larger ε → more lenient, larger neighborhoods

2. **min_samples (MinPts)**: The minimum number of points needed to form a dense region
   - Think of it as the "density threshold"
   - Smaller MinPts → easier to form clusters
   - Larger MinPts → requires denser regions

### Three Types of Points [Paper §4.1, p. 227]

DBSCAN classifies every point into one of three categories:

- **Core Point**: Has at least MinPts neighbors within distance ε (including itself)
  - These form the "backbone" of clusters
  - Dense region centers

- **Border Point**: Within ε distance of a core point, but doesn't have enough neighbors to be core
  - These are on the "edges" of clusters
  - Belong to a cluster but aren't dense enough to be core

- **Noise Point**: Neither core nor border
  - Outliers that don't belong to any cluster
  - Points in sparse regions

---

## 2. Mathematical Formalization <a id='mathematical-formalization'></a>

Now let's formalize these concepts mathematically [Paper §4.1, p. 227].

### Epsilon-Neighborhood

The **ε-neighborhood** of a point p is defined as:

```
N_ε(p) = {q ∈ D | dist(p, q) ≤ ε}
```

Where:
- **D** is the database (dataset) of points
- **dist(p, q)** is the distance function (typically Euclidean distance)
- **ε** (epsilon) is the radius parameter

**In plain English**: N_ε(p) is the set of all points within distance ε from point p.

### Core Point Definition

A point p is a **core point** if:

```
|N_ε(p)| ≥ MinPts
```

Where:
- **|N_ε(p)|** denotes the cardinality (size) of the ε-neighborhood
- **MinPts** is the minimum number of points required

**In plain English**: A point is core if it has at least MinPts neighbors (including itself) within distance ε.

### Distance Function

The default distance metric is **Euclidean distance** [Paper §4.1, p. 227]:

For 2D points p = (x₁, y₁) and q = (x₂, y₂):

```
dist(p, q) = √((x₁ - x₂)² + (y₁ - y₂)²)
```

For n-dimensional points:

```
dist(p, q) = √(Σᵢ(pᵢ - qᵢ)²)
```

### Cluster Formation

A **cluster** C is a maximal set of density-connected points [Paper §4.1, p. 227]:

1. All points in C are density-connected to each other
2. No point outside C is density-connected to any point in C

**Note**: We'll explore density-connectivity in detail in `04_density_concepts.ipynb`. For now, understand that clusters are formed by connecting core points and their neighbors.

---

## 3. Setup and Imports <a id='setup'></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles, make_blobs
import sys
sys.path.append('..')

from src.dbscan_from_scratch import DBSCAN
from src.data_loader import load_sample_data
from src.visualization import DBSCANVisualizer

# Set random seed for reproducibility
np.random.seed(42)

# Initialize visualizer
viz = DBSCANVisualizer()

print("✓ Setup complete!")

---

## 4. Basic DBSCAN Application <a id='basic-application'></a>

Let's start with a simple example using the "moons" dataset - two crescent-shaped clusters.

### Load and Visualize Data

In [ ]:
# Generate moons dataset
X, _ = make_moons(n_samples=300, noise=0.05, random_state=42)

# Visualize original data
plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], c='gray', alpha=0.6, s=50)
plt.title('Original Data: Two Crescent Moons', fontsize=14, fontweight='bold')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Dataset shape: {X.shape}")
print(f"Number of points: {len(X)}")

### Apply DBSCAN

Now let's apply DBSCAN with reasonable parameters:
- **eps = 0.3**: Points within 0.3 units are considered neighbors
- **min_samples = 5**: Need at least 5 points to form a dense region

In [ ]:
# Create and fit DBSCAN model
dbscan = DBSCAN(eps=0.3, min_pts=5)
labels = dbscan.fit_predict(X)

# Analyze results
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)
n_core = len(dbscan.get_core_points())

print(f"\n📊 DBSCAN Results:")
print(f"  • Number of clusters found: {n_clusters}")
print(f"  • Number of noise points: {n_noise}")
print(f"  • Number of core points: {n_core}")
print(f"  • Total points: {len(X)}")

# Visualize clustering results with enhanced annotations
viz.plot_clusters(X, labels, title="DBSCAN Clustering Results (ε=0.3, MinPts=5)")
plt.annotate('Cluster 1', xy=(0.5, 0.5), fontsize=12, fontweight='bold', 
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
plt.annotate('Cluster 2', xy=(1.5, -0.5), fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
plt.show()

### Understanding the Results

In the plot above:
- **Different colors** represent different clusters
- **Black 'x' markers** are noise points (outliers)
- DBSCAN successfully identified the two crescent-shaped clusters!

**Key Observation**: DBSCAN handles non-circular shapes perfectly. This is one of its major advantages over algorithms like K-Means [Paper §2, p. 226].

---

## 5. Understanding Point Types <a id='point-types'></a>

Let's visualize the three types of points DBSCAN identifies [Paper §4.1, p. 227].

In [ ]:
# Visualize point type classification
viz.plot_point_types(
    X, 
    labels, 
    dbscan.core_sample_indices_, 
    eps=0.3,
    title="DBSCAN Point Type Classification (ε=0.3, MinPts=5)"
)
plt.show()

# Calculate point type statistics
core_mask = np.zeros(len(X), dtype=bool)
core_mask[dbscan.core_sample_indices_] = True
noise_mask = labels == -1
border_mask = ~core_mask & ~noise_mask

print("\n📌 Point Type Breakdown:")
print(f"  • Core points (blue): {np.sum(core_mask)} ({100*np.sum(core_mask)/len(X):.1f}%)")
print(f"    - Form the 'skeleton' of clusters")
print(f"    - Have |N_ε(p)| ≥ MinPts")
print(f"\n  • Border points (green): {np.sum(border_mask)} ({100*np.sum(border_mask)/len(X):.1f}%)")
print(f"    - On the edges of clusters")
print(f"    - Within ε of a core point but not core themselves")
print(f"\n  • Noise points (black): {np.sum(noise_mask)} ({100*np.sum(noise_mask)/len(X):.1f}%)")
print(f"    - Outliers that don't belong to any cluster")
print(f"    - Neither core nor border")

### Examining a Specific Point's Neighborhood

Let's look at the ε-neighborhood of a specific core point to understand the concept better [Paper §4.1, p. 227].

In [ ]:
# Select a core point to examine
core_point_idx = dbscan.core_sample_indices_[0]

# Visualize its epsilon-neighborhood
viz.plot_epsilon_neighborhood(X, core_point_idx, eps=0.3, labels=labels)
plt.show()

# Calculate neighborhood size
distances = np.sqrt(np.sum((X - X[core_point_idx])**2, axis=1))
neighbors = np.where(distances <= 0.3)[0]

print(f"\nAnalyzing Point {core_point_idx}:")
print(f"  Coordinates: {X[core_point_idx]}")
print(f"  |N_ε(p)|: {len(neighbors)} points")
print(f"  MinPts threshold: 5")
print(f"  Is core point: {len(neighbors) >= 5} ✓")
print(f"  Point type: {dbscan.get_point_type(core_point_idx).name}")

---

## 6. Parameter Effects <a id='parameter-effects'></a>

Understanding how eps and min_samples affect clustering is crucial for applying DBSCAN effectively [Paper §5.1, p. 229].

### Effect of Epsilon (ε)

Let's see how different epsilon values affect the clustering:

In [ ]:
# Test different eps values
eps_values = [0.15, 0.3, 0.5, 0.8]
min_pts = 5

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, eps in zip(axes, eps_values):
    dbscan_test = DBSCAN(eps=eps, min_pts=min_pts)
    labels_test = dbscan_test.fit_predict(X)
    
    n_clusters = len(set(labels_test)) - (1 if -1 in labels_test else 0)
    n_noise = list(labels_test).count(-1)
    
    # Plot
    unique_labels = set(labels_test)
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    
    for label, color in zip(unique_labels, colors):
        if label == -1:
            mask = labels_test == label
            ax.scatter(X[mask, 0], X[mask, 1], c='black', marker='x', s=50, alpha=0.5)
        else:
            mask = labels_test == label
            ax.scatter(X[mask, 0], X[mask, 1], c=[color], s=50, alpha=0.7)
    
    ax.set_title(f'ε = {eps}, MinPts = {min_pts}\n{n_clusters} clusters, {n_noise} noise points', 
                fontsize=11, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🎯 Epsilon Effects:")
print("  • ε too small (0.15): Many small clusters, lots of noise")
print("  • ε just right (0.3): Correct clustering")
print("  • ε moderate (0.5): Clusters start to merge")
print("  • ε too large (0.8): All points in one cluster")

### Effect of MinPts

Now let's see how MinPts affects the clustering:

In [ ]:
# Test different min_pts values
eps = 0.3
minpts_values = [3, 5, 10, 15]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, min_pts in zip(axes, minpts_values):
    dbscan_test = DBSCAN(eps=eps, min_pts=min_pts)
    labels_test = dbscan_test.fit_predict(X)
    
    n_clusters = len(set(labels_test)) - (1 if -1 in labels_test else 0)
    n_noise = list(labels_test).count(-1)
    n_core = len(dbscan_test.get_core_points())
    
    # Plot
    unique_labels = set(labels_test)
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    
    for label, color in zip(unique_labels, colors):
        if label == -1:
            mask = labels_test == label
            ax.scatter(X[mask, 0], X[mask, 1], c='black', marker='x', s=50, alpha=0.5)
        else:
            mask = labels_test == label
            ax.scatter(X[mask, 0], X[mask, 1], c=[color], s=50, alpha=0.7)
    
    ax.set_title(f'ε = {eps}, MinPts = {min_pts}\n{n_clusters} clusters, {n_core} core, {n_noise} noise', 
                fontsize=11, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🎯 MinPts Effects:")
print("  • MinPts too small (3): More points become core, possibly over-clustering")
print("  • MinPts just right (5): Good balance")
print("  • MinPts moderate (10): Fewer core points, more noise")
print("  • MinPts too large (15): Very strict density requirement, many noise points")

---

## 7. Different Dataset Shapes <a id='dataset-shapes'></a>

One of DBSCAN's key strengths is handling arbitrary cluster shapes [Paper §2, p. 226]. Let's test it on various geometries.

### Concentric Circles

In [ ]:
# Generate circles dataset
X_circles, _ = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

# Apply DBSCAN
dbscan_circles = DBSCAN(eps=0.2, min_pts=5)
labels_circles = dbscan_circles.fit_predict(X_circles)

# Visualize
viz.plot_clusters(X_circles, labels_circles, title="DBSCAN on Concentric Circles (ε=0.2, MinPts=5)")
plt.show()

n_clusters = len(set(labels_circles)) - (1 if -1 in labels_circles else 0)
print(f"\nCircles Dataset:")
print(f"  • Clusters found: {n_clusters}")
print(f"  • Noise points: {list(labels_circles).count(-1)}")
print(f"  • Success: DBSCAN correctly identifies both circular clusters!")

### Spherical Blobs

In [ ]:
# Generate blobs dataset
X_blobs, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.5, random_state=42)

# Apply DBSCAN
dbscan_blobs = DBSCAN(eps=0.8, min_pts=5)
labels_blobs = dbscan_blobs.fit_predict(X_blobs)

# Visualize
viz.plot_clusters(X_blobs, labels_blobs, title="DBSCAN on Spherical Blobs (ε=0.8, MinPts=5)")
plt.show()

n_clusters = len(set(labels_blobs)) - (1 if -1 in labels_blobs else 0)
print(f"\nBlobs Dataset:")
print(f"  • Clusters found: {n_clusters}")
print(f"  • Noise points: {list(labels_blobs).count(-1)}")
print(f"  • Note: DBSCAN works well on spherical clusters too!")

### Varying Density Clusters

In [ ]:
# Generate dataset with varying densities
np.random.seed(42)
X_dense = np.random.randn(100, 2) * 0.3 + [0, 0]  # Dense cluster
X_sparse = np.random.randn(100, 2) * 0.8 + [3, 3]  # Sparse cluster
X_varied = np.vstack([X_dense, X_sparse])

# Apply DBSCAN
dbscan_varied = DBSCAN(eps=0.5, min_pts=5)
labels_varied = dbscan_varied.fit_predict(X_varied)

# Visualize
viz.plot_clusters(X_varied, labels_varied, title="DBSCAN on Varying Density (ε=0.5, MinPts=5)")
plt.annotate('Dense cluster', xy=(0, 0), fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
plt.annotate('Sparse cluster', xy=(3, 3), fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8))
plt.show()

n_clusters = len(set(labels_varied)) - (1 if -1 in labels_varied else 0)
print(f"\nVarying Density Dataset:")
print(f"  • Clusters found: {n_clusters}")
print(f"  • Noise points: {list(labels_varied).count(-1)}")
print(f"  • Challenge: Single eps value may not work well for all densities")
print(f"  • Note: Consider OPTICS algorithm for varying density (see advanced notebooks)")

---

## 8. Exercises <a id='exercises'></a>

Test your understanding with these hands-on exercises!

### Exercise 1: Parameter Selection (Beginner)

**Task**: Given the dataset below, experiment with different eps and min_samples values to find parameters that produce exactly 3 clusters.

**Hint**: Start with eps around 0.5-1.0 and min_samples around 3-7.

In [ ]:
# Exercise 1: Dataset
X_ex1, _ = make_blobs(n_samples=200, centers=3, cluster_std=0.6, random_state=10)

# Visualize the data
plt.figure(figsize=(8, 6))
plt.scatter(X_ex1[:, 0], X_ex1[:, 1], c='gray', alpha=0.6)
plt.title('Exercise 1: Find parameters for 3 clusters')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)
plt.show()

# TODO: Try different parameters here
# eps = ???
# min_pts = ???
# dbscan_ex1 = DBSCAN(eps=eps, min_pts=min_pts)
# labels_ex1 = dbscan_ex1.fit_predict(X_ex1)
# viz.plot_clusters(X_ex1, labels_ex1, title=f"Your Solution (ε={eps}, MinPts={min_pts})")
# plt.show()

<details>
<summary><b>Click to reveal Exercise 1 solution</b></summary>

**Solution**:

In [ ]:
# Exercise 1 Solution
eps = 0.7
min_pts = 5

dbscan_ex1 = DBSCAN(eps=eps, min_pts=min_pts)
labels_ex1 = dbscan_ex1.fit_predict(X_ex1)

n_clusters = len(set(labels_ex1)) - (1 if -1 in labels_ex1 else 0)
n_noise = list(labels_ex1).count(-1)

viz.plot_clusters(X_ex1, labels_ex1, title=f"Solution (ε={eps}, MinPts={min_pts})")
plt.show()

print(f"\nSolution Results:")
print(f"  • Clusters found: {n_clusters}")
print(f"  • Noise points: {n_noise}")
print(f"\nNote: Other parameter combinations may also work!")
print(f"      For example: eps=0.8, min_pts=4 or eps=0.6, min_pts=6")

</details>

### Exercise 2: Point Type Analysis (Intermediate)

**Task**: For the moons dataset with eps=0.3 and min_samples=5, calculate:
1. The percentage of points that are core points
2. The percentage of points that are border points
3. The percentage of points that are noise

**Hint**: Use the `get_core_points()` method and the labels array.

In [ ]:
# Exercise 2: Your code here
X_ex2, _ = make_moons(n_samples=200, noise=0.05, random_state=42)
dbscan_ex2 = DBSCAN(eps=0.3, min_pts=5)
labels_ex2 = dbscan_ex2.fit_predict(X_ex2)

# TODO: Calculate percentages
# core_percentage = ???
# border_percentage = ???
# noise_percentage = ???

<details>
<summary><b>Click to reveal Exercise 2 solution</b></summary>

**Solution**:

In [ ]:
# Exercise 2 Solution
X_ex2, _ = make_moons(n_samples=200, noise=0.05, random_state=42)
dbscan_ex2 = DBSCAN(eps=0.3, min_pts=5)
labels_ex2 = dbscan_ex2.fit_predict(X_ex2)

# Identify point types
core_indices = dbscan_ex2.get_core_points()
core_mask = np.zeros(len(X_ex2), dtype=bool)
core_mask[core_indices] = True

noise_mask = labels_ex2 == -1
border_mask = ~core_mask & ~noise_mask

# Calculate percentages
total = len(X_ex2)
core_percentage = 100 * np.sum(core_mask) / total
border_percentage = 100 * np.sum(border_mask) / total
noise_percentage = 100 * np.sum(noise_mask) / total

print("\nExercise 2 Solution:")
print(f"  • Core points: {np.sum(core_mask)} ({core_percentage:.1f}%)")
print(f"  • Border points: {np.sum(border_mask)} ({border_percentage:.1f}%)")
print(f"  • Noise points: {np.sum(noise_mask)} ({noise_percentage:.1f}%)")
print(f"  • Total: {total} (100.0%)")

# Visualize
viz.plot_point_types(X_ex2, labels_ex2, core_indices, eps=0.3)
plt.show()

</details>

### Exercise 3: Epsilon-Neighborhood Calculation (Advanced)

**Task**: Given the following 5 points and ε = 1.5, manually calculate which points are in the ε-neighborhood of Point A (index 0). Then verify your answer with code.

Points:
- A: (0, 0)
- B: (1, 0)
- C: (0, 1)
- D: (2, 0)
- E: (1.5, 1.5)

**Hint**: Use the Euclidean distance formula: d = √((x₁-x₂)² + (y₁-y₂)²)

In [ ]:
# Exercise 3: Your calculations and code here
points_ex3 = np.array([
    [0, 0],    # A
    [1, 0],    # B
    [0, 1],    # C
    [2, 0],    # D
    [1.5, 1.5] # E
])
point_names = ['A', 'B', 'C', 'D', 'E']
eps_ex3 = 1.5

# TODO: Calculate distances and identify neighbors
# Your manual calculations:
# dist(A, B) = ???
# dist(A, C) = ???
# dist(A, D) = ???
# dist(A, E) = ???

<details>
<summary><b>Click to reveal Exercise 3 solution</b></summary>

**Solution**:

In [ ]:
# Exercise 3 Solution
points_ex3 = np.array([
    [0, 0],    # A
    [1, 0],    # B
    [0, 1],    # C
    [2, 0],    # D
    [1.5, 1.5] # E
])
point_names = ['A', 'B', 'C', 'D', 'E']
eps_ex3 = 1.5
point_A_idx = 0

print("Manual Calculations:")
print("  dist(A, A) = √((0-0)² + (0-0)²) = 0.0 ≤ 1.5 ✓")
print("  dist(A, B) = √((0-1)² + (0-0)²) = 1.0 ≤ 1.5 ✓")
print("  dist(A, C) = √((0-0)² + (0-1)²) = 1.0 ≤ 1.5 ✓")
print("  dist(A, D) = √((0-2)² + (0-0)²) = 2.0 > 1.5 ✗")
print("  dist(A, E) = √((0-1.5)² + (0-1.5)²) = √4.5 ≈ 2.12 > 1.5 ✗")
print("\nN_ε(A) = {A, B, C}")
print("|N_ε(A)| = 3")

# Verify with code
print("\n\nCode Verification:")
distances = np.sqrt(np.sum((points_ex3 - points_ex3[point_A_idx])**2, axis=1))
for i, (name, dist) in enumerate(zip(point_names, distances)):
    in_neighborhood = "✓" if dist <= eps_ex3 else "✗"
    print(f"  dist(A, {name}) = {dist:.2f} {in_neighborhood}")

neighbors = np.where(distances <= eps_ex3)[0]
print(f"\nN_ε(A) = {{{', '.join([point_names[i] for i in neighbors])}}}")
print(f"|N_ε(A)| = {len(neighbors)}")

# Visualize
plt.figure(figsize=(8, 8))
plt.scatter(points_ex3[:, 0], points_ex3[:, 1], c='blue', s=200, zorder=5)
for i, name in enumerate(point_names):
    plt.annotate(name, (points_ex3[i, 0], points_ex3[i, 1]), 
                xytext=(8, 8), textcoords='offset points', fontsize=14, fontweight='bold')

# Highlight neighbors
neighbors_no_self = neighbors[neighbors != point_A_idx]
if len(neighbors_no_self) > 0:
    plt.scatter(points_ex3[neighbors_no_self, 0], points_ex3[neighbors_no_self, 1],
               facecolors='none', edgecolors='orange', s=400, linewidths=3, zorder=4)

# Draw epsilon circle
circle = plt.Circle((points_ex3[point_A_idx, 0], points_ex3[point_A_idx, 1]), eps_ex3,
                   color='red', fill=False, linestyle='--', linewidth=2, alpha=0.5)
plt.gca().add_patch(circle)

plt.scatter([points_ex3[point_A_idx, 0]], [points_ex3[point_A_idx, 1]], 
           c='red', s=300, marker='*', zorder=6, label='Point A')
plt.xlim(-1, 3)
plt.ylim(-1, 3)
plt.xlabel('X')
plt.ylabel('Y')
plt.title(f'ε-neighborhood of A (ε = {eps_ex3})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

</details>

---

## 9. Summary <a id='summary'></a>

Congratulations! You've completed the DBSCAN basics notebook. Let's review what you've learned.

### Key Concepts

1. **Density-Based Clustering** [Paper §1, p. 226]:
   - Clusters are dense regions of points
   - Separated by regions of lower density
   - No need to specify number of clusters in advance

2. **Two Parameters** [Paper §4.1, p. 227]:
   - **eps (ε)**: Maximum distance for neighborhood
   - **min_samples (MinPts)**: Minimum points for dense region

3. **Three Point Types** [Paper §4.1, p. 227]:
   - **Core**: |N_ε(p)| ≥ MinPts
   - **Border**: Within ε of core but not core itself
   - **Noise**: Neither core nor border (outliers)

4. **Mathematical Foundation** [Paper §4.1, p. 227]:
   - ε-neighborhood: N_ε(p) = {q ∈ D | dist(p, q) ≤ ε}
   - Core point condition: |N_ε(p)| ≥ MinPts
   - Euclidean distance: d(p,q) = √(Σᵢ(pᵢ-qᵢ)²)

5. **Advantages**:
   - Handles arbitrary cluster shapes
   - Automatic outlier detection
   - Deterministic results
   - Intuitive parameters

6. **Parameter Effects**:
   - **eps too small**: Many small clusters, lots of noise
   - **eps too large**: Clusters merge together
   - **MinPts too small**: Over-clustering, many core points
   - **MinPts too large**: Under-clustering, many noise points

### What You Can Do Now

✓ Understand the core concepts of DBSCAN  
✓ Apply DBSCAN to various dataset types  
✓ Interpret clustering results (core, border, noise)  
✓ Understand parameter effects on clustering  
✓ Recognize when DBSCAN is appropriate  
✓ Visualize and analyze DBSCAN outcomes  

### Limitations to Be Aware Of

1. **Varying Density**: Single eps value may not work well for clusters with different densities
2. **High Dimensions**: Distance metrics become less meaningful in high dimensions ("curse of dimensionality")
3. **Parameter Selection**: Choosing good eps and MinPts requires domain knowledge or experimentation
4. **Computational Cost**: O(n²) for naive implementation (can be improved with spatial indexing)

**Note**: Some of these limitations are addressed by extensions like OPTICS (covered in advanced notebooks).

---

## 10. Next Steps <a id='next-steps'></a>

Ready to dive deeper? Here's your recommended learning path:

### Immediate Next Steps

1. **[02_mathematical_foundations.ipynb](02_mathematical_foundations.ipynb)**
   - Formal mathematical definitions
   - Rigorous treatment of ε-neighborhoods
   - Distance metrics and their properties
   - Mathematical proofs and properties

2. **[06_parameter_tuning.ipynb](06_parameter_tuning.ipynb)** (currently 02_eps_minpts_tuning.ipynb)
   - Systematic parameter selection methods
   - K-distance graph technique [Paper §5.1]
   - Elbow method for epsilon selection
   - Interactive parameter exploration

3. **[03_epsilon_neighborhood.ipynb](03_epsilon_neighborhood.ipynb)**
   - Interactive ε-neighborhood exploration
   - Visual understanding of neighborhoods
   - Parameter sensitivity analysis

### For Deeper Understanding

4. **[04_density_concepts.ipynb](04_density_concepts.ipynb)**
   - Density-reachability [Paper §4.1]
   - Density-connectivity [Paper §4.1]
   - How clusters are formed

5. **[05_algorithm_walkthrough.ipynb](05_algorithm_walkthrough.ipynb)**
   - Step-by-step algorithm execution
   - Algorithm visualization
   - Understanding the clustering process

6. **[07_comparing_algorithms.ipynb](07_comparing_algorithms.ipynb)** (currently 03_comparing_kmeans_dbscan.ipynb)
   - DBSCAN vs K-Means
   - DBSCAN vs OPTICS
   - When to use which algorithm

### Documentation Resources

- **[docs/01_theory_and_math.md](../docs/01_theory_and_math.md)**: Comprehensive theory
- **[docs/02_density_concepts.md](../docs/02_density_concepts.md)**: Density-based concepts
- **[docs/04_parameter_tuning.md](../docs/04_parameter_tuning.md)**: Parameter selection guide
- **[docs/00_how_to_read_the_paper.md](../docs/00_how_to_read_the_paper.md)**: Paper reading guide

### Implementation

- **[src/dbscan_from_scratch.py](../src/dbscan_from_scratch.py)**: Complete implementation
- **[src/visualization.py](../src/visualization.py)**: Visualization functions

### Original Paper

- **[1996-DBSCAN-KDD.pdf](../1996-DBSCAN-KDD.pdf)**: The foundational paper

---

## References

**Primary Reference:**

Ester, M., Kriegel, H. P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. In *Proceedings of the Second International Conference on Knowledge Discovery and Data Mining (KDD-96)* (Vol. 96, No. 34, pp. 226-231).

**Key Sections Referenced:**
- §1 (p. 226): Introduction and motivation
- §2 (p. 226): Related work and comparison with other algorithms
- §4.1 (p. 227): Density-based notions of clusters - formal definitions
- §4.2 (p. 228): DBSCAN algorithm description
- §5.1 (p. 229): Parameter determination

---

**Happy Clustering! 🎯**

*For questions or issues, refer to the repository README or documentation.*